<a href="https://colab.research.google.com/github/weagan/Encoder-Decoder/blob/main/FLAN_T5_XXL_translation_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FLAN-T5-XXL (11B) Translation Notebook (Colab)

This notebook loads **Hugging Face `google/flan-t5-xxl` (11B)** and runs translation plus lightweight tests.

> **Runtime recommendation:** Colab GPU with high VRAM (A100 40GB preferred).
> This notebook pins the model to GPU (`device_map={'': 0}`) to avoid recent `bitsandbytes` auto-offload errors.

In [ ]:
# Verify runtime
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. FLAN-T5-XXL may be too large for CPU-only execution.')

In [ ]:
# Install dependencies (run once per fresh Colab runtime)
!pip -q install -U "transformers>=4.45" accelerate bitsandbytes sentencepiece sacrebleu

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

MODEL_ID = 'google/flan-t5-xxl'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading model (this can take a while)...')
# Use the supported quantization API and pin the model to GPU to avoid
# ValueError about CPU/disk dispatch in recent transformers+bitsandbytes builds.
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    device_map={'': 0},
    quantization_config=bnb_config,
)

print('Model loaded.')

In [ ]:
def translate(text: str, source_lang: str, target_lang: str, max_new_tokens: int = 96) -> str:
    """Translate text with FLAN-T5 using an instruction prompt."""
    prompt = f"Translate from {source_lang} to {target_lang}: {text}"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
# Quick demo
sample = 'The weather is beautiful today.'
translated = translate(sample, 'English', 'French')
print('Source:', sample)
print('French:', translated)

## Lightweight Translation Tests

These are practical quality checks (not full benchmarking). They validate:
1. Output is non-empty.
2. Output differs from source text.
3. Basic corpus-level metric (`chrF`) is reported for a tiny test set.

In [ ]:
from sacrebleu.metrics import CHRF

TEST_CASES = [
    {'src': 'Hello, how are you?', 'ref': 'Bonjour, comment allez-vous ?'},
    {'src': 'I love reading books about science.', 'ref': "J'aime lire des livres sur la science."},
    {'src': 'Please turn left at the next street.', 'ref': 'Veuillez tourner à gauche à la prochaine rue.'},
]

preds, refs = [], []
for i, case in enumerate(TEST_CASES, start=1):
    pred = translate(case['src'], 'English', 'French')
    preds.append(pred)
    refs.append(case['ref'])

    print(f"[{i}] SRC: {case['src']}")
    print(f"    PRED: {pred}")
    print(f"    REF : {case['ref']}\n")

    assert isinstance(pred, str) and len(pred.strip()) > 0, 'Empty translation output'
    assert pred.strip().lower() != case['src'].strip().lower(), 'Output identical to source'

chrf = CHRF(word_order=2)
score = chrf.corpus_score(preds, [refs])
print(f"chrF++ score: {score.score:.2f}")

assert score.score > 25, f'chrF++ too low: {score.score:.2f}'
print('All tests passed.')

## Optional: Translate custom text

Edit `text`, `source_lang`, and `target_lang`.

In [ ]:
text = 'Machine learning can help translate languages efficiently.'
source_lang = 'English'
target_lang = 'Spanish'

print(translate(text, source_lang, target_lang))